# Пересечение кластеризаций и итоговая сегментация точек

Завершающий ноутбук пайплайна. Здесь ничего не считается заново - берутся результаты кластеризации по
размеру (`02`), цене (`03`/`04`) и ассортименту (`05`) и сводятся вместе двумя способами:

1. **Хитмапы пересечений** - сколько точек попадает в каждую пару кластеров (например: сколько точек с
   "премиальным" ценовым профилем при этом относятся к "маленькому" размерному кластеру). Показывает,
   насколько разные срезы кластеризации согласуются друг с другом или, наоборот, дают разную информацию.
2. **Итоговая ручная сегментация** - на основе этих пересечений точки были вручную сведены в 19 бизнес-понятных
   сегментов (`Итоговый кластер`: "Классическая аптека", "ВИП-аптека", "Аптека у дома", "Ларьки" и т.д.) -
   и дальше эта итоговая сегментация проверяется на согласованность с брендом сети, локацией, форматом выкладки
   и другими характеристиками точки.

Входные файлы со сведёнными результатами кластеризации (с реальными точками и брендами) в портфолио не
включены - здесь показана сама техника сведения и проверки, а не воспроизводимый расчёт. Названия брендов
сети и внутренних бизнес-единиц (БЕ) заменены на условные "Бренд N" / "БЕ N" - важен сам факт кластеризации
по ним, а не конкретные названия.

## Импорт библиотек

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import textwrap

pd.options.display.max_columns = 100

## Настройка параметров

In [ ]:
# Порядок ценовых кластеров (от самого премиального к самому экономичному) - подобран вручную по
# средней цене внутри каждого кластера, чтобы на графиках сегменты шли в осмысленном порядке.
PRICE_CLUSTER_ORDER = ['SuperPremium_2', 'SuperPremium', 'Premium', 'Medium', 'MediumLow', 'Economy', 'Economy_0']

# Итоговая сегментация точек - 19 бизнес-понятных сегментов, результат ручного сведения кластеров
# по цене/размеру/ассортименту. Порядок - от крупных/премиальных форматов к мелким/экономичным.
FINAL_SEGMENT_ORDER = [
    'Центральная аптека', 'Классическая аптека', 'Классическая маленькая аптека', 'Классическая эконом аптека',
    'ВИП-аптека', 'Маленькая ВИП-аптека',
    'Премиум-Супермаркет', 'Премиум-Минимаркет',
    'Супермаркет', 'Минимаркет', 'Эконом-Супермаркет',
    'Аптека у дома', 'Маленькая аптека у дома', 'Эконом аптека у дома', 'Ларьки',
    'Сельская центральная аптека', 'Сельская эконом-аптека',
    'Спец. ассортимент', 'ЛПУ',
]

BRAND_PALETTE_10 = ['#0000FF', '#800000', '#FF0000', '#000080', '#00FF00', '#FF8C00',
                    '#DC143C', '#48D1CC', '#4B0082', '#808080']
BE_PALETTE_9 = ['#0000FF', '#FF00FF', '#4B0082', '#000080', '#00FF00', '#FF8C00', '#FF0000', '#800000', '#48D1CC']

## Функции

In [ ]:
def plot_intersection_heatmap(pivot: pd.DataFrame, row_order=None, col_order=None,
                               row_labels=None, col_labels=None, xlabel='', ylabel='', figsize=(30, 9)):
    """
    Хитмап пересечения двух кластеризаций: pivot - таблица "кластер A x кластер B -> число точек"
    (обычно pd.pivot_table с aggfunc='count'). row_order/col_order переставляют строки/столбцы в заданном
    порядке (например, от премиального к экономичному) - без этого порядок кластеров был бы произвольным
    (по номеру, присвоенному при кластеризации, который сам по себе ничего не значит).
    """
    to_plot = pivot
    if row_order is not None:
        to_plot = to_plot.iloc[row_order]
    if col_order is not None:
        to_plot = to_plot.iloc[:, col_order]

    if row_labels is not None or col_labels is not None:
        to_plot = pd.DataFrame(np.array(to_plot),
                                index=row_labels if row_labels is not None else to_plot.index,
                                columns=col_labels if col_labels is not None else to_plot.columns)

    fig = plt.figure(figsize=figsize)
    ax = sns.heatmap(to_plot, cmap="YlGnBu", annot=True, square=True, annot_kws={"size": 20}, fmt='g')
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.show()


def plot_segment_breakdown(df: pd.DataFrame, hue_col: str, hue_order=None, palette=None, figsize=(17, 11)):
    """Swarmplot выручки по итоговым сегментам (FINAL_SEGMENT_ORDER), с разбивкой по указанной характеристике точки."""
    fig = plt.figure(figsize=figsize)
    ax = sns.swarmplot(x='Итоговый кластер', y='Revenue', hue=hue_col, data=df, size=7,
                        order=FINAL_SEGMENT_ORDER, hue_order=hue_order, palette=palette)
    ax.set_xticklabels(textwrap.fill(x.get_text(), 12) for x in ax.get_xticklabels())
    plt.xticks(rotation=45)
    plt.show()

## Алгоритм выполнения

### Пересечения кластеров по размеру / цене / ассортименту

Сводный файл с результатами всех кластеризаций по каждой точке (в портфолио не включён). Сравнение идёт
на удалённой от одного конкретного производителя выборке (см. `04_..._fassovka.ipynb` про причину исключения) -
это финальная версия расчёта, более ранняя версия с этим производителем в данных не показана.

In [ ]:
all_clusters = pd.read_excel('сводная_кластеризация.xlsx', 'все_метрики')
intersections = all_clusters[['TradePointCode', 'ID_groups', 'ID_prices', 'ID_size_square', 'ID_size_rev_cheque']]

In [ ]:
# цены x ассортимент - порядок столбцов (ID_groups) подобран вручную по итоговой сегментации (05), чтобы
# похожие по духу ассортиментные кластеры шли рядом
ASSORT_ORDER = [17, 11, 16, 4, 1, 14, 0, 6, 8, 13, 3, 12, 10, 5, 15, 9, 7, 2]

pivot = intersections.pivot_table(index='ID_prices', columns='ID_groups', values='TradePointCode', aggfunc='count', fill_value=0)
plot_intersection_heatmap(pivot, row_order=[4, 6, 3, 1, 0, 2, 5], col_order=ASSORT_ORDER,
                          row_labels=PRICE_CLUSTER_ORDER, col_labels=ASSORT_ORDER,
                          xlabel='Group cluster', ylabel='Price cluster')

In [ ]:
# размер (площадь) x ассортимент
pivot = intersections.pivot_table(index='ID_size_square', columns='ID_groups', values='TradePointCode', aggfunc='count', fill_value=0)
plot_intersection_heatmap(pivot, row_order=[0, 5, 3, 2, 1, 4], col_order=ASSORT_ORDER,
                          xlabel='Group cluster', ylabel='Square Size cluster', figsize=(30, 10))

In [ ]:
# размер (выручка/чеки) x ассортимент
pivot = intersections.pivot_table(index='ID_size_rev_cheque', columns='ID_groups', values='TradePointCode', aggfunc='count', fill_value=0)
plot_intersection_heatmap(pivot, row_order=[1, 4, 3, 2, 5, 0], col_order=ASSORT_ORDER,
                          xlabel='Group cluster', ylabel='Revenue-Cheque Size cluster', figsize=(30, 10))

In [ ]:
# два способа мерить "размер точки" (площадь vs выручка/чеки) между собой
pivot = intersections.pivot_table(index='ID_size_rev_cheque', columns='ID_size_square', values='TradePointCode', aggfunc='count', fill_value=0)
plot_intersection_heatmap(pivot, row_order=[1, 4, 3, 2, 5, 0], col_order=[4, 1, 2, 3, 5, 0],
                          xlabel='Square Size cluster', ylabel='Revenue-Cheque Size cluster', figsize=(30, 10))

In [ ]:
# цена x размер (оба варианта размера)
pivot = intersections.pivot_table(index='ID_prices', columns='ID_size_square', values='TradePointCode', aggfunc='count', fill_value=0)
plot_intersection_heatmap(pivot, row_order=[4, 6, 3, 1, 0, 2, 5], col_order=[4, 1, 2, 3, 5, 0],
                          row_labels=PRICE_CLUSTER_ORDER, xlabel='Square Size cluster', ylabel='Price cluster', figsize=(30, 10))

pivot = intersections.pivot_table(index='ID_prices', columns='ID_size_rev_cheque', values='TradePointCode', aggfunc='count', fill_value=0)
plot_intersection_heatmap(pivot, row_order=[4, 6, 3, 1, 0, 2, 5], col_order=[0, 5, 2, 3, 4, 1],
                          row_labels=PRICE_CLUSTER_ORDER, xlabel='Revenue-Cheque Size cluster', ylabel='Price cluster', figsize=(30, 10))

### Итоговая (человеко-понятная) сегментация

Файл с уже вручную сведённой итоговой сегментацией по каждой точке (в портфолио не включён - содержит
реальные точки, бренды и регионы). Дальше - серия проверок: сходится ли итоговая сегментация с тем, что и так
известно про точку (бренд сети, локация, формат выкладки, действующая категория точки, статус ВИП/дискаунтер,
регион/бизнес-единица) - то есть выглядит ли сегментация правдоподобно, а не просто как артефакт кластеризации.

In [ ]:
final = pd.read_excel('финальная_кластеризация.xlsx', 'все_метрики')
final = final.rename({'STORE_LOCATION_LVL_NM1': 'БЕ'}, axis=1)

In [ ]:
plot_segment_breakdown(final, hue_col='BrandName',
                        hue_order=[f'Бренд {i}' for i in range(1, 11)], palette=BRAND_PALETTE_10)

In [ ]:
plot_segment_breakdown(final, hue_col='Локация')

In [ ]:
plot_segment_breakdown(final, hue_col='Выкладка')

Разбивка по действующей категории точки - два варианта справочника категорий (полная версия и упрощённая "ПФ").

In [ ]:
plot_segment_breakdown(final, hue_col='Текущая категория', hue_order=['1', '2', '3', '4', '5', '6', 'Закрыто'])

In [ ]:
plot_segment_breakdown(final, hue_col='Текущая категория ПФ',
                        hue_order=['A1', 'A2', 'B1', 'B2', 'C1', 'C2', 'D1', 'D2', 'E', 'Закрыто', 'Инд.АП'])

In [ ]:
plot_segment_breakdown(final, hue_col='Дискаунтер')

In [ ]:
plot_segment_breakdown(final, hue_col='ВИП', hue_order=['нет', 'да'])

In [ ]:
plot_segment_breakdown(final, hue_col='БЕ',
                        hue_order=[f'БЕ {i}' for i in range(1, 10)], palette=BE_PALETTE_9)

### Пересечения именованных кластеров между собой

Те же хитмапы пересечений, что в начале ноутбука, но уже по укрупнённым человеко-понятным меткам
(`Ценовой кластер`: эконом/медиум/премиум, `Размерный кластер`: большие/средние/маленькие) вместо
исходных числовых ID - проще читать глазами.

In [ ]:
pivot = final.pivot_table(index='Ценовой кластер', columns='Размерный кластер', values='TradePointCode', aggfunc='count', fill_value=0)
plot_intersection_heatmap(pivot, row_order=[1, 0, 2], col_order=[1, 2, 0],
                          xlabel='Размерный кластер', ylabel='Ценовой кластер', figsize=(20, 8))

In [ ]:
pivot = final.pivot_table(index='Ценовой кластер', columns='Текущая категория ПФ', values='TradePointCode', aggfunc='count', fill_value=0)
plot_intersection_heatmap(pivot, row_order=[1, 0, 2],
                          xlabel='Текущая категория ПФ', ylabel='Ценовой кластер', figsize=(20, 8))

In [ ]:
pivot = final.pivot_table(index='Ценовой кластер', columns='Итоговый кластер', values='TradePointCode', aggfunc='count', fill_value=0)
pivot = pivot.loc[['Эконом', 'Медиум', 'Премиум'], FINAL_SEGMENT_ORDER]
plot_intersection_heatmap(pivot, xlabel='Итоговый кластер', ylabel='Ценовой кластер', figsize=(32, 8))

In [ ]:
pivot = final.pivot_table(index='Размерный кластер', columns='Итоговый кластер', values='TradePointCode', aggfunc='count', fill_value=0)
pivot = pivot.loc[['Большие аптеки', 'Средние аптеки', 'Маленькие аптеки'], FINAL_SEGMENT_ORDER]
plot_intersection_heatmap(pivot, xlabel='Итоговый кластер', ylabel='Размерный кластер', figsize=(32, 8))

In [ ]:
pivot = final.pivot_table(index='Текущая категория', columns='Итоговый кластер', values='TradePointCode', aggfunc='count', fill_value=0)
pivot = pivot.loc[:, FINAL_SEGMENT_ORDER]
plot_intersection_heatmap(pivot, xlabel='Итоговый кластер', ylabel='Текущая категория', figsize=(34, 12))

pivot = final.pivot_table(index='Текущая категория ПФ', columns='Итоговый кластер', values='TradePointCode', aggfunc='count', fill_value=0)
pivot = pivot.loc[:, FINAL_SEGMENT_ORDER]
plot_intersection_heatmap(pivot, xlabel='Итоговый кластер', ylabel='Текущая категория ПФ', figsize=(34, 12))